### Forest Area Calculation within the Biodiversity Exploritories

In [31]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
from shapely.ops import transform
from shapely.geometry import mapping
import warnings
warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)

import pystac
from pystac_client import Client # for STAC API access
import rioxarray # to work on remote and local rasters
import pyproj

In [32]:
# Read in Data and Shapefiles
alb = gpd.read_file("Boarders/Exploratorium_alb.gpkg")
hai = gpd.read_file("Boarders/Exploratorium_hai.gpkg")
sch = gpd.read_file("Boarders/Exploratorium_sch.gpkg")

In [33]:
# define the url for the STAC API
api_url = "https://earth-search.aws.element84.com/v1"
# Create a STAC client
client = Client.open(api_url)
# Collections
collections = client.get_collections()
# Print collection names
for collection in collections:
    print(collection)

<CollectionClient id=sentinel-2-pre-c1-l2a>
<CollectionClient id=cop-dem-glo-30>
<CollectionClient id=naip>
<CollectionClient id=cop-dem-glo-90>
<CollectionClient id=landsat-c2-l2>
<CollectionClient id=sentinel-2-l2a>
<CollectionClient id=sentinel-2-l1c>
<CollectionClient id=sentinel-2-c1-l2a>
<CollectionClient id=sentinel-1-grd>


With in Earth Search we can access sentinal, copernicus and some landsat. I will use Sentinal 2 because it has 10 m x 10 m resolution and flies over every 5 days

The message  
```
Error retrieving number of items found: {"message": "Internal server error"}
```
means that the STAC API server could not process your request.  
This is not a problem with your Python code, but with the API or the data you are sending.  
Possible reasons include:
- The geometry you provided is too complex or too large for the API to handle.
- The API server is experiencing issues or is temporarily unavailable.
- The search parameters (such as the date range or area) are too broad and return too many results.

**How to troubleshoot:**
- Try simplifying your geometry (e.g., use a smaller area or reduce the number of vertices).
- Try a shorter date range.
- Check the API status or try again later.
- Make sure your geometry is valid GeoJSON.

Yes, you should check that the CRS (Coordinate Reference System) of your geometry matches the CRS expected by the STAC collection (usually EPSG:4326, WGS84).  
If your geometry is not in EPSG:4326, you should reproject it before sending it to the API.  
Below is an example of how to check and reproject if needed.

In [37]:
#help(client.search)

In [35]:
# Check and reproject to EPSG:4326 if necessary
if hai.crs != "EPSG:4326":
    hai = hai.to_crs("EPSG:4326")

# Now extract the bounding box from the reprojected geometry
minx, miny, maxx, maxy = hai.total_bounds
bbox_geom = {
    "type": "Polygon",
    "coordinates": [[
        [minx, miny],
        [minx, maxy],
        [maxx, maxy],
        [maxx, miny],
        [minx, miny]
    ]]
}

collection_sentinel_2_l2a = "sentinel-2-l2a"
search = client.search(
    collections=collection_sentinel_2_l2a,
    intersects=bbox_geom,
    datetime="2025-05-01/2025-05-31",
    query=['eo:cloud_cover<1']  # Limit the number of items returned
)

try:
    print(f'Number of items found: {search.matched()}')
except Exception as e:
    print("Error retrieving number of items found:", e)

Number of items found: 5


Only 5 images in the month of May match the criteria

In [36]:
# Save to JSON file
items = search.item_collection()
items.save_object("Hai_Sentinel")